# MicroDiptera v0.4 — high-resolution hierarchical pilot

Family-balanced iNaturalist pilot → 518 px whole image + tiles → specimen fusion → conditional family/genus/species heads → centroid open-set gate. Species heads use only A/B labels by default; iNaturalist Research Grade is not silently promoted to cryptic-species truth.

In [ ]:
REPO_URL = 'https://github.com/SaniyaSani/EntoKey.git'
PER_FAMILY = 40       # increase only after this pilot succeeds
IMAGE_SIZE = 518      # divisible by DINOv2 patch size 14
TILE_GRID = 2         # whole specimen + 4 detail tiles
MODEL_DIR = 'models_microdiptera'
TEST_IMAGE_AFTER_TRAINING = False  # change to True only when you want the upload dialog


In [ ]:
from pathlib import Path
import os, subprocess, sys
project = Path('/content/entokey-v04')
if project.exists():
    subprocess.run(['git', '-C', str(project), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(project)], check=True)
os.chdir(project)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Project ready:', project)


In [ ]:
subprocess.run([sys.executable, 'scripts/build_microdiptera_pilot.py', '--per-family', str(PER_FAMILY), '--download'], check=True)
subprocess.run([sys.executable, 'scripts/embed_multiview.py', '--manifest', 'data/microdiptera/microdiptera_manifest.csv', '--out-dir', MODEL_DIR, '--image-size', str(IMAGE_SIZE), '--tile-grid', str(TILE_GRID), '--batch-size', '4'], check=True)
subprocess.run([sys.executable, 'scripts/train_hierarchical.py', '--model-dir', MODEL_DIR, '--min-family', '12', '--min-genus', '6', '--min-species', '4'], check=True)
subprocess.run([sys.executable, 'scripts/build_retrieval_index.py', '--model-dir', MODEL_DIR], check=True)
print('MicroDiptera v0.4 pilot trained:', MODEL_DIR)


In [ ]:
from google.colab import files
archive = 'microdiptera_v04_models.zip'
subprocess.run(['zip', '-qr', archive, MODEL_DIR], check=True)
files.download(archive)


## Test one unseen fly
Run this cell separately after training. A missing species result is a scientifically valid abstention when no curated A/B species head exists.

In [ ]:
if not TEST_IMAGE_AFTER_TRAINING:
    print('Model saved. Set TEST_IMAGE_AFTER_TRAINING=True and run this cell to test a fly.')
else:
    import io, json
    import pandas as pd
    from PIL import Image
    from IPython.display import display
    sys.path.insert(0, str(project / 'src'))
    from diptera_id.embedding import DINOEmbedder
    from diptera_id.hierarchy import load_classifier_bundle
    from diptera_id.retrieval import RetrievalIndex
    uploaded = files.upload()
    filename, raw = next(iter(uploaded.items()))
    image = Image.open(io.BytesIO(raw)).convert('RGB')
    display(image)
    bundle = load_classifier_bundle(Path(MODEL_DIR) / 'classifiers.joblib')
    cfg = bundle.metadata.get('embedding', {})
    embedder = DINOEmbedder(cfg.get('backbone', 'facebook/dinov2-small'), image_size=int(cfg.get('image_size', 518)))
    embedding = embedder.embed_multicrop(image, tile_grid=int(cfg.get('tile_grid', 2)), include_whole=bool(cfg.get('include_whole', True)))
    predictions = bundle.predict_all(embedding, top_k=5)
    for rank, candidates in predictions.items():
        print('\n' + rank.upper())
        for item in candidates:
            similarity = item.get('centroid_similarity')
            print(f"{item['taxon']:<32} score={item['probability']:.1%} centroid={similarity:.3f}")
    print('\nOPEN SET:', bundle.last_open_set)
    neighbours = RetrievalIndex.load(MODEL_DIR).search(embedding, k=5)
    display(pd.DataFrame(neighbours)[['similarity', 'family', 'genus', 'species', 'image_url']])
